# Chapter 17 — Tools Produce Context

## Question

**What does a tool cost in context before it runs and after it runs?**

Falsifiable structure: with identical underlying capability and data, do different capability surfaces and different observation shapings produce different trajectory token totals? If yes, the capability surface and the observation surface are independent design variables. No selection accuracy is simulated; only occupancy differences are established.

## Setup — tools as two surfaces

Each tool has a definition side (name, description, schema, examples, definition tokens) and an execution side (call tokens, result tokens, completeness, continuation). Token counts are fixtures.

In [ ]:
from dataclasses import dataclass
from enum import Enum

class Completeness(Enum):
    COMPLETE = 'COMPLETE'
    FILTERED = 'FILTERED'
    PAGINATED = 'PAGINATED'
    TRUNCATED = 'TRUNCATED'
    REFERENCE_ONLY = 'REFERENCE_ONLY'
    ERROR = 'ERROR'

@dataclass(frozen=True)
class ToolDef:
    name: str
    description_tokens: int
    schema_tokens: int
    example_tokens: int = 0

    @property
    def definition_tokens(self):
        return self.description_tokens + self.schema_tokens + self.example_tokens

RUNTIME_TOOLS = ['read', 'search', 'run_tests', 'shell', 'lint', 'deploy',
                 'db_query', 'issues', 'chat', 'calendar', 'drive', 'monitor']
TASK_TOOLS = ['read', 'search', 'run_tests', 'lint']
print(f'runtime knows {len(RUNTIME_TOOLS)} tools; the task needs {len(TASK_TOOLS)}.')

## Baseline — existence is not residency, execution is not admission

In [ ]:
exposed_all = list(RUNTIME_TOOLS)
print(f'tools exist in runtime: {len(RUNTIME_TOOLS)}; definitions in context: {len(exposed_all)} (baseline: all)')
assert len(RUNTIME_TOOLS) == 12
raw_records, decisive = 200, 199  # one decisive record near the tail
print(f'backend holds {raw_records} records; decisive record at index {decisive}')
assert decisive == raw_records - 1

## Intervention 1 — standing capability cost across three exposures

In [ ]:
DEFS = {name: ToolDef(name, 120, 380) for name in RUNTIME_TOOLS}
DEFS['db_query'] = ToolDef('db_query', 120, 2380)  # one enormous schema
DISCOVERY_SURFACE = 300

all12 = sum(DEFS[t].definition_tokens for t in RUNTIME_TOOLS)
task4 = sum(DEFS[t].definition_tokens for t in TASK_TOOLS)
discovery = DISCOVERY_SURFACE + sum(DEFS[t].definition_tokens for t in ['search', 'read'])
print(f'all 12 resident:              {all12} tokens')
print(f'task-specific 4 resident:     {task4} tokens')
print(f'discovery surface + 2 loaded: {discovery} tokens')
assert all12 > task4 > discovery
assert len(RUNTIME_TOOLS) != len(TASK_TOOLS)
print('Availability, representation, and execution are three separate gates.')

## Intervention 2 — count is not weight

In [ ]:
tiny = [ToolDef(f't{i}', 40, 60) for i in range(10)]
tiny_total = sum(t.definition_tokens for t in tiny)
huge = DEFS['db_query'].definition_tokens
print(f'10 tiny schemas: {tiny_total} tokens; 1 enormous schema: {huge} tokens')
assert huge > tiny_total
print('Tool count is not tool context cost.')

## Intervention 3 — observation shaping over identical data

One deterministic backend, 200 records, decisive record last. Five shapings; only structural properties measured.

In [ ]:
RECORD_TOKENS, PAGE, ANCHOR = 200, 20, 60
TOTAL_RAW = raw_records * RECORD_TOKENS

shaped = {
    'RAW': {'tokens': TOTAL_RAW, 'required_present': True, 'identifier_present': True,
              'continuation': False, 'completeness': Completeness.COMPLETE},
    'FILTERED': {'tokens': 12 * RECORD_TOKENS, 'required_present': True, 'identifier_present': True,
                 'continuation': True, 'completeness': Completeness.FILTERED},
    'PAGINATED': {'tokens': PAGE * RECORD_TOKENS, 'required_present': False, 'identifier_present': False,
                  'continuation': True, 'completeness': Completeness.PAGINATED},
    'ANCHOR_PLUS_REFERENCE': {'tokens': ANCHOR, 'required_present': False, 'identifier_present': True,
                              'continuation': True, 'completeness': Completeness.REFERENCE_ONLY},
    'TRUNCATED': {'tokens': 10 * RECORD_TOKENS, 'required_present': False, 'identifier_present': False,
                  'continuation': True, 'completeness': Completeness.TRUNCATED},
}
print(f"{'shaping':20s} {'tokens':>6s} {'required':>8s} {'ident':>5s} {'continues':>9s}")
for name, s in shaped.items():
    print(f"{name:20s} {s['tokens']:6d} {str(s['required_present']):>8s} {str(s['identifier_present']):>5s} {str(s['continuation']):>9s}")
assert shaped['RAW']['required_present'] is True
assert shaped['TRUNCATED']['required_present'] is False
assert shaped['PAGINATED']['continuation'] is True
assert shaped['ANCHOR_PLUS_REFERENCE']['identifier_present'] is True
print('Hard truncation loses the tail; pagination preserves the path; anchor preserves identity.')

# Completeness is first-class metadata: identical emptiness, different meanings.
empty_complete, empty_truncated, empty_filtered = [], [], []
print('[] + COMPLETE means nothing matched; [] + TRUNCATED/FILTERED means absence is not evidence.')

## Intervention 4 — trajectory ledger and errors

ILLUSTRATIVE TRAJECTORY — NOT MODEL-SELECTION EVIDENCE. Suite A: larger definitions, fewer calls, focused observations. Suite B: smaller definitions, retries, larger cumulative results. Call counts are synthetic and deterministic.

In [ ]:
suite_a = {'definitions_per_turn': 400, 'turns': 6, 'calls': 2, 'result_tokens': 3000, 'errors': 0}
suite_b = {'definitions_per_turn': 100, 'turns': 6, 'calls': 5, 'result_tokens': 9000, 'errors': 2}
cost_a = suite_a['definitions_per_turn'] * suite_a['turns'] + suite_a['result_tokens']
cost_b = suite_b['definitions_per_turn'] * suite_b['turns'] + suite_b['result_tokens']
print(f'suite A trajectory: {cost_a} tokens; suite B trajectory: {cost_b} tokens')
assert cost_a < cost_b
print('Standing cost + marginal cost = trajectory cost. Either side optimised alone can lose.')

# Errors as observations: dump versus concise, decisive fields retained in both framings.
concise_error = {'tool': 'deploy', 'error_type': 'invalid parameter', 'message': 'region',
                 'diagnostic': 'stray year appended to query', 'continuation': 'retry without year'}
print('concise error keeps:', concise_error)
assert all(k in concise_error for k in ('tool', 'error_type', 'message', 'diagnostic', 'continuation'))
print('Diagnostic evidence is never suppressed for token savings.')

## Observation — producer-side versus consumer-side reduction

In [ ]:
print(f'producer-side filtering (FILTERED): {shaped["FILTERED"]["tokens"]} tokens before candidacy')
print(f'consumer-side admission would start from RAW: {shaped["RAW"]["tokens"]} tokens')
assert shaped['FILTERED']['tokens'] < shaped['RAW']['tokens']
print('Filtering before the window and admission after it are different boundaries. Not collapsed.')

## Try it

1. Move the decisive record to index 5 and re-score TRUNCATED: the hazard is positional, not intrinsic.
2. Double every schema and confirm exposure rankings never change — only the standing totals do.
3. Give PAGINATED ten follow-up pages and price the accumulation against one FILTERED call.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# print(DEFS['db_query'])

## What this demonstrates

- Tools have a standing context cost before execution and a marginal context cost after execution.
- The capability surface and the observation surface are independent design variables.
- Result completeness is itself context metadata: identical emptiness means different things under different flags.

## What this does not demonstrate

- That fewer exposed tools improve real selection, or that dynamic discovery wins.
- That filtering or pagination always saves tokens, or that concise schemas improve behaviour.
- That synthetic call counts predict real trajectories.
- That tool permissions and context admission are the same problem.

## Connection to the chapter

The right observation is chosen; one question remains about the form it takes:

> Even after we choose the right tool observation to admit, one final question remains: in what representation should the information enter the context?

That is Chapter 18.